# Animal speed data cleaning

Cleans the messy animal dataset used by Biodiversity Hub's **Species Speed** chart (Feature 3).

Run the cells top to bottom in Google Colab. When prompted, upload the raw `messy_animals.csv`
(the unmodified export of the shared Google Sheet). The notebook keeps only the `name`, `speed`
and `diet` columns, cleans them entirely in code, and exports a CSV that is committed to the web app
as `public/sample_animals.csv`.

In [3]:
# word2number is not preinstalled in Colab; it converts number words like "fifty-six" to 56
%pip install -q word2number

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re

import numpy as np
import pandas as pd
from word2number import w2n

In [2]:
# In Colab this opens a file picker for the raw CSV; outside Colab it falls back to a local copy next to the notebook
try:
    from google.colab import files

    RAW_PATH = next(iter(files.upload()))
except ImportError:
    RAW_PATH = os.path.join(os.path.dirname(os.path.abspath("clean_animal_data.ipynb")), "messy_animals.csv")

raw = pd.read_csv(RAW_PATH, dtype=str)
print(f"Loaded {RAW_PATH}: {raw.shape[0]} rows x {raw.shape[1]} columns")
raw[["Animal", "Average Speed (km/h)", "Diet"]].head(8)

Loaded /home/ankit/f26-eng-r2-deliverable/data-cleaning/messy_animals.csv: 205 rows x 17 columns


,Animal,Average Speed (km/h),Diet
0,Aardvark,40,NaN
1,Aardwolf,24-30,NaN
2,African Elephant,25,Herbivore
3,African Lion,58,Carnivore
4,African Wild Dog,fifty-six,Carnivore
5,Alpine Ibex,56-64,Herbivore
6,Amazon Rainforest Frog,0.1-1,Insectivore
7,American Bison,40-56,Herbivore


In [3]:
KEEP_DIETS = {"carnivore", "herbivore", "omnivore"}


def fix_encoding(value):
    """Repair names that were double-encoded in the sheet (e.g. "GalÃ¡pagos" -> "Galápagos"); leave everything else as is."""
    try:
        return str(value).encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value


def parse_speed(value):
    """Turn a messy speed cell into a number in km/h, or NaN when it holds no usable value."""
    text = re.sub(r"\(.*?\)", "", str(value)).strip()  # drop notes such as "(in water)"
    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        return float(text)
    if re.fullmatch(r"\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?", text):  # a range like "40-64" becomes its average
        return np.mean([float(n) for n in re.findall(r"\d+(?:\.\d+)?", text)])
    try:
        return float(w2n.word_to_num(text.lower()))  # number words like "fifty-six"
    except ValueError:
        return np.nan  # "Not Applicable", "Varies", ...


def parse_diet(value):
    """Standardize diet to carnivore/herbivore/omnivore using the first listed diet; anything else becomes NaN."""
    primary = str(value).strip().lower().split(",")[0].strip()
    return primary if primary in KEEP_DIETS else np.nan


# quick sanity checks on the helpers
print([fix_encoding(v) for v in ["GalÃ¡pagos Penguin", "Galápagos Tortoise", "African Lion"]])
print([parse_speed(v) for v in ["40", "40-64", "fifty-six", "16-24 (in water)", "Not Applicable", "Varies"]])
print([parse_diet(v) for v in ["Carnivore", "  Herbivore   ", "Carnivore, Piscivore", "Insectivore", "Omnivore, Herbivore"]])

['Galápagos Penguin', 'Galápagos Tortoise', 'African Lion']
[40.0, np.float64(52.0), 56.0, np.float64(20.0), nan, nan]
['carnivore', 'herbivore', 'carnivore', nan, 'omnivore']


In [4]:
# Keep only the three columns the chart needs and give them simple names (the raw file itself is never edited)
df = raw[["Animal", "Average Speed (km/h)", "Diet"]].rename(
    columns={"Animal": "name", "Average Speed (km/h)": "speed", "Diet": "diet"}
)
df = df.apply(lambda col: col.str.strip()).replace("", np.nan)  # trim whitespace; blanks become missing
df = df.dropna(subset=["name", "speed", "diet"])  # drop rows missing any of the three fields
df["name"] = df["name"].map(fix_encoding)  # repair double-encoded accents
df["speed"] = df["speed"].map(parse_speed)  # words -> numbers, ranges -> averages
df["diet"] = df["diet"].map(parse_diet)  # keep only carnivore / herbivore / omnivore
df = df.dropna(subset=["speed", "diet"]).drop_duplicates(subset="name", keep="first")
df["speed"] = df["speed"].round(2)
df = df.sort_values("name").reset_index(drop=True)

print(f"{len(raw)} raw rows -> {len(df)} cleaned rows")
print(df["diet"].value_counts().to_string())
print(f"speed range: {df['speed'].min()} - {df['speed'].max()} km/h")
df.head(10)

205 raw rows -> 144 cleaned rows
diet
carnivore    60
herbivore    49
omnivore     35
speed range: 0.02 - 120.0 km/h


,name,speed,diet
0,African Elephant,25.0,herbivore
1,African Lion,58.0,carnivore
2,African Wild Dog,56.0,carnivore
3,Alpine Ibex,60.0,herbivore
4,American Bison,48.0,herbivore
5,Arabian Horse,65.0,herbivore
6,Arabian Oryx,55.0,herbivore
7,Arctic Fox,60.0,omnivore
8,Arowana,24.0,carnivore
9,Asian Elephant,40.0,herbivore


In [5]:
OUTPUT = "Ankit Biswas - Cleaned Animal Data.csv"
df.to_csv(OUTPUT, index=False)

# In Colab, download the cleaned file; outside Colab, also write it straight into the web app's public folder
try:
    from google.colab import files

    files.download(OUTPUT)
except ImportError:
    df.to_csv("../public/sample_animals.csv", index=False)

print(f"Wrote {OUTPUT} ({len(df)} rows). Commit it to the web app as public/sample_animals.csv.")

Wrote Ankit Biswas - Cleaned Animal Data.csv (144 rows). Commit it to the web app as public/sample_animals.csv.


## Cleaning rules (and why)

- **Columns**: only `Animal`, `Average Speed (km/h)` and `Diet` are used, renamed to `name`, `speed`, `diet` in code. The raw CSV is never edited by hand, so re-running this notebook on the original export gives the same output.
- **Whitespace**: every value is trimmed (the diet column has lots of padded entries such as `"           Carnivore"`).
- **Missing values**: rows with an empty name, speed or diet are dropped.
- **Speed**: plain numbers are kept; ranges such as `40-64` become their average (52); number words such as `fifty-six` are converted with `word2number`; notes in parentheses such as `(in water)` are ignored; `Not Applicable` / `Varies` cannot be turned into a number and those rows are dropped.
- **Diet**: lower-cased and reduced to the first listed diet, and only `carnivore`, `herbivore` and `omnivore` are kept. `Carnivore, Piscivore` therefore counts as carnivore, while `Insectivore`, `Piscivore` and `Scavenger` rows are dropped because the chart only compares the three requested categories.
- **Duplicates**: a few animals appear twice in the raw sheet; the first occurrence is kept so each bar in the chart is a unique animal.

Result: 205 raw rows become 144 clean rows (60 carnivores, 49 herbivores, 35 omnivores).